# 第 19 课｜风险中性概率

> **数据身份**：固定随机种子生成的模拟教学数据，不是真实证券行情。  
> **环境**：`pkm-quant` · Python 3.11 · 默认离线运行。  
> **课程原始材料**：`19_youtube-ETWnhDFQEGs/index.html`  
> **所属模块**：P1-M4 · 随机模型与衍生品风险

## 本节量化问题

风险中性概率是市场上涨预测，还是无套利定价工具？

## 准确定义与策略报告位置

风险中性测度把可交易资产的期望增长率调整为无风险利率，用贴现期望复制衍生品价格。

- **核心关系**：一周期二叉树风险中性概率 q=(e^(rΔt)-d)/(u-d)；价格=e^(-rΔt)E_Q[payoff]。
- **写入报告**：定价测度、复制组合和无套利边界
- **P2 衔接**：P2 第 33 课：矩阵与递推计算。


## 运行环境与数据契约

先确认项目、数据目录和数据身份。后续联网数据源必须通过适配层进入，不能改变本 Notebook 的研究计算口径。

In [1]:
from pathlib import Path
import json
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "data" / "fixtures").exists():
    ROOT = ROOT.parent
DATA_DIR = ROOT / "data" / "fixtures"
OUTPUT_DIR = ROOT / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

RANDOM_SEED = 365
print("项目:", ROOT)
print("数据:", DATA_DIR)
print("数据身份: 固定随机种子生成的模拟教学数据，不是真实证券行情")


项目: /Users/mojiawen/Documents/学习资料归档/_暂存/personal_knowledge_manager/量化交易/实验/p1-quant-foundations
数据: /Users/mojiawen/Documents/学习资料归档/_暂存/personal_knowledge_manager/量化交易/实验/p1-quant-foundations/data/fixtures
数据身份: 固定随机种子生成的模拟教学数据，不是真实证券行情


## 一周期二叉树风险中性定价

先检查 `d < exp(rΔt) < u`，否则存在简单无套利冲突。风险中性 q 用于贴现复制，不是上涨预测。

In [2]:
spot, strike = 100.0, 100.0
up, down = 1.20, 0.85
rate, dt = 0.03, 1.0

growth = np.exp(rate * dt)
assert down < growth < up
q = (growth - down) / (up - down)

payoff_up = max(spot * up - strike, 0.0)
payoff_down = max(spot * down - strike, 0.0)
option_price = np.exp(-rate * dt) * (q * payoff_up + (1.0 - q) * payoff_down)

display(pd.Series({"risk_neutral_q": q, "payoff_up": payoff_up, "payoff_down": payoff_down, "call_price": option_price}))

risk_neutral_q     0.515584
payoff_up         20.000000
payoff_down        0.000000
call_price        10.006931
dtype: float64

## 方法边界与阶段验收

风险中性概率不是现实上涨概率；结论依赖无套利、可交易和复制假设。

完成本课后应能：

1. 不看课程材料准确解释核心概念和公式符号；
2. 用小样本手工核对 Notebook 的关键结果；
3. 说明结果能支持什么判断、不能支持什么判断；
4. 指出它在策略报告中的位置，以及仍缺少的数据、成本或稳健性证据。
